In [1]:
!git clone https://github.com/aitf-its-tim3-dfk/david.git

Cloning into 'david'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 49 (delta 19), reused 38 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 1.33 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [1]:
%cd david

/content/david


In [2]:
!git switch feat/refactor-farhan

M	config.yaml
M	dataset.py
M	encoder.py
Already on 'feat/refactor-farhan'
Your branch is up to date with 'origin/feat/refactor-farhan'.


In [5]:
%pip install ftfy regex decord scikit-learn yacs

In [7]:
import kagglehub
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
%env KAGGLE_USERNAME=$username
%env KAGGLE_KEY=$key

env: KAGGLE_USERNAME=farhanwew
env: KAGGLE_KEY=51376389df8a5a095fd393132ef1b88f


In [8]:
!kaggle datasets download farhanwew/real-vs-gen-videos

Dataset URL: https://www.kaggle.com/datasets/farhanwew/real-vs-gen-videos
License(s): CC0-1.0
100% 44.6G/44.6G [07:06<00:00, 112MB/s]



In [9]:
!unzip real-vs-gen-videos.zip -d final_dataset


  inflating: final_dataset/real/coin/videos/49_X3F1xP3LU4o.mp4  
  inflating: final_dataset/real/coin/videos/4_MKBcbI1M1eE.mp4  
  inflating: final_dataset/real/coin/videos/4_MR_olQaxdoQ.mp4  
  inflating: final_dataset/real/coin/videos/4_Nj1aD-3jLhU.mp4  
  inflating: final_dataset/real/coin/videos/4_Qg8NfR5jotQ.mp4  
  inflating: final_dataset/real/coin/videos/4_UML9pfjeRzs.mp4  
  inflating: final_dataset/real/coin/videos/4_aBywINg8JB4.mp4  
  inflating: final_dataset/real/coin/videos/4_zYUo59uj4Bw.mp4  
  inflating: final_dataset/real/coin/videos/50_IzO0bqSz3cA.mp4  
  inflating: final_dataset/real/coin/videos/51_t4PWyvokS28.mp4  
  inflating: final_dataset/real/coin/videos/52_HYfF0X5JPLA.mp4  
  inflating: final_dataset/real/coin/videos/52_Nwmn9Sqm5oY.mp4  
  inflating: final_dataset/real/coin/videos/52_SvAuKcwH4_w.mp4  
  inflating: final_dataset/real/coin/videos/52_Wvrj0Dr4S8A.mp4  
  inflating: final_dataset/real/coin/videos/52_ljrC0UF4e8c.mp4  
  inflating: final_dataset/real

In [13]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1Nx30Kbu5xnv6dPwz4I3Ivy380LCdp1Md

Downloading...
From (original): https://drive.google.com/uc?id=1Nx30Kbu5xnv6dPwz4I3Ivy380LCdp1Md
From (redirected): https://drive.google.com/uc?id=1Nx30Kbu5xnv6dPwz4I3Ivy380LCdp1Md&confirm=t&uuid=59998914-f9c0-434d-8b45-52ba71c68e10
To: /content/david/k400_clip_complete_finetuned_30_epochs.pth
100% 1.62G/1.62G [00:27<00:00, 59.0MB/s]


In [4]:
%%writefile config.yaml

# DAViD Configuration
# All fields are optional — missing keys fall back to defaults in config.py.
#
# ── Legacy Google Drive setup (pre-refactor) ──────────────────────────────────
# To revert to the old structure (Drive mount + JSON cache instead of CSV),
# replace the paths section with the following and call get_train_val_loaders()
# without csv_path:
#
#   paths:
#     base_dir: /content/drive/MyDrive/data mining gemastik
#     best_model_save_path: best_detector_model.pt
#     vificlip_checkpoint: /content/drive/MyDrive/vifi_clip_30_epochs_k400_full_finetuned.pth
#
# Then copy the cache locally in the notebook before training:
#   CACHE_PATH       = "video_train_10000_cache_fixed_2.json"
#   DRIVE_CACHE_PATH = "/content/drive/MyDrive/data mining gemastik/video train 10000/video_train_10000_cache_fixed_2.json"
#   if not os.path.exists(CACHE_PATH):
#       shutil.copy(DRIVE_CACHE_PATH, CACHE_PATH)
#
# Real video directories (legacy):
#   /content/drive/.shortcut-targets-by-id/1Wcbv564DV62urzCJvYmvnkDD_Z74ZKLa/GenVideo-Val/Real
#   /content/drive/MyDrive/data mining gemastik/K4/videos_val
#
# Fake video directories (legacy):
#   /content/drive/MyDrive/gemastik-datmin/pika/train_pika
#   /content/drive/MyDrive/data mining gemastik/Sora/train_OpenSora
#   /content/drive/MyDrive/data mining gemastik/SVD/train_SVD
# ─────────────────────────────────────────────────────────────────────────────

paths:
  base_dir: final_dataset
  metadata_csv: final_dataset/metadata.csv
  best_model_save_path: best_detector_model.pt
  vificlip_checkpoint: k400_clip_complete_finetuned_30_epochs.pth

model:
  clip_arch: ViT-B/16
  class_names: ["true", "false"]
  head_type: deep          # "simple" (1-layer) | "deep" (3-layer)

training:
  learning_rate: 1.0e-3
  batch_size: 64
  num_epochs: 3
  num_frames: 16
  num_workers: 4
  input_dim: 512
  num_classes: 1
  use_amp: true            # mixed precision — disable if running on CPU
  lr_scheduler: cosine     # "cosine" | "plateau" | null
  patience: 3              # early stopping epochs; null to disable

dataset:
  val_split: 0.2           # used when train_size and val_size are both null
  train_size: 20000         # e.g. 15000 — null means use all remaining after val
  val_size: 1000        # e.g. 200   — null means use val_split ratio


Overwriting config.yaml


In [5]:
from config import CLIP_ARCH, CLASS_NAMES, VIFICLIP_CHECKPOINT
from encoder import load_feature_extractor

feature_extractor = load_feature_extractor(
    arch=CLIP_ARCH,
    class_names=CLASS_NAMES,
    checkpoint_path=VIFICLIP_CHECKPOINT,
).cuda()

In [6]:
import os
from config import METADATA_CSV, BASE_DIR, BATCH_SIZE, VAL_SPLIT, NUM_WORKERS
from config import TRAIN_SIZE, VAL_SIZE
from config import INPUT_DIM, NUM_CLASSES, LEARNING_RATE, NUM_EPOCHS, BEST_MODEL_SAVE_PATH, HEAD_TYPE

In [8]:
from model import build_transform
from dataset import get_train_val_loaders
from train import set_seed

set_seed(42)

CLEAN_CSV = "final_dataset/metadata_clean.csv"

preprocess = build_transform(224)
train_loader, val_loader, val_files = get_train_val_loaders(
    transform=preprocess,
    csv_path=METADATA_CSV,
    base_dir=BASE_DIR,
    val_split=VAL_SPLIT,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    validate=True,
    clean_csv_path=CLEAN_CSV,  # save clean CSV once, reuse next time
)

Validating videos: 100%|██████████| 77658/77658 [20:42<00:00, 62.48it/s]


Loaded 75402 videos from final_dataset/metadata.csv (2256 broken files skipped)
Clean CSV saved: final_dataset/metadata_clean.csv
Dataset split: 20000 training files, 1000 validation files.


In [9]:
from config import USE_AMP, LR_SCHEDULER, PATIENCE
from train import set_seed, run_training

set_seed(42)

classifier = run_training(
    feature_extractor=feature_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    head_type=HEAD_TYPE,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    save_path=BEST_MODEL_SAVE_PATH,
    use_amp=USE_AMP,
    lr_scheduler=LR_SCHEDULER,
    patience=PATIENCE,
)

Using device: cuda | AMP: True | Scheduler: cosine | Patience: 3
Creating classification head: 'deep'

--- Epoch 1/3 | Training ---


100%|██████████| 313/313 [30:36<00:00,  5.87s/it]


Average Training Loss: 0.2763
--- Epoch 1/3 | Validation ---


100%|██████████| 16/16 [01:44<00:00,  6.53s/it]


Average Validation Loss: 0.1076
Validation Accuracy: 96.50%
LR: 7.50e-04
New best model saved with accuracy: 96.50%

--- Epoch 2/3 | Training ---


...

Average Training Loss: 0.0582
--- Epoch 3/3 | Validation ---


100%|██████████| 16/16 [01:44<00:00,  6.51s/it]


Average Validation Loss: 0.0703
Validation Accuracy: 97.50%
LR: 0.00e+00

--- Training Complete ---
Loading best model from best_detector_model.pt for final report...


100%|██████████| 16/16 [01:44<00:00,  6.52s/it]

Average Validation Loss: 0.0980
Validation Accuracy: 96.70%

--- Final Performance Report (on validation set) ---
              precision    recall  f1-score   support

    Fake (0)       0.97      0.98      0.97       633
    Real (1)       0.96      0.95      0.95       367

    accuracy                           0.97      1000
   macro avg       0.97      0.96      0.96      1000
weighted avg       0.97      0.97      0.97      1000



In [10]:
from evaluate import evaluate
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics = evaluate(classifier, feature_extractor, val_loader, val_files, device)

EVALUATION REPORT


Evaluating: 100%|██████████| 16/16 [01:44<00:00,  6.53s/it]


AUC-ROC : 0.9944
Optimal threshold : 0.270  (F1=0.9637  vs  F1@0.5=0.9615)

── Classification Report (threshold=0.5) ──
              precision    recall  f1-score   support

    Fake (0)       0.97      0.98      0.98       633
    Real (1)       0.97      0.95      0.96       367

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000

── Confusion Matrix (threshold=0.5) ──
              Pred Fake  Pred Real
  True Fake         622         11
  True Real          17        350

  False Positive Rate : 0.017  (real misclassified as fake)
  False Negative Rate : 0.046  (fake slipping through)

── Per-source Accuracy ──
  K4              n=  265   acc=95.5%   auc=nan
  MSSRVT          n=   99   acc=94.9%   auc=nan
  SD              n=  135   acc=99.3%   auc=nan
  SVD             n=  123   acc=97.6%   auc=nan
  Sora            n=  120   acc=100.0%   auc=nan
  ZeroScope       n=  11


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dis